<a href="https://colab.research.google.com/github/delaneydow/ncl_diffusion/blob/main/00_setup_patch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup patch  (run once, after `Project_setup.ipynb`)

Your setup notebook is a good base. This one fixes a few things that would stop the next steps from running,
and adds what we need for the **diffusion vs. GAN** comparison. It is safe to re-run.

**Fixes**
1. `os.makedirs` list had a misplaced quote (`"ncl_diffusion," "ncl_diffusion/data"`), which creates a folder literally named `ncl_diffusion,ncl_diffusion`.
2. `backbones.py` and `system.py` import `..compat`, but the file you wrote is `model_imports.py`, so the package could not be imported. `compat.py` is now written (and `model_imports.py` just re-exports it).
3. Dependencies: dropped `monai-generative` (MONAI >= 1.4 has `DiffusionModelUNet` and the schedulers in core), added `monai>=1.4`, `einops`, `torchmetrics`, `torch-fidelity` (FID).
4. The registries printed empty because `__init__.py` does not import the sub-packages. Notebooks now do `from ncl_diffusion import data, models, train`.

**Additions**
- `GanCfg` section in `config.py` (so GAN hyper-parameters live in `config.json` like everything else)
- `models/gan.py`: label-conditional GAN baseline (ResNet G, projection D, hinge loss, spectral norm, DiffAugment)
- `train/gan.py`: training loop with EMA, checkpoints, resume, previews (same conventions as `train/diffusion.py`)
- a smoke test at the end that trains both models for 1 epoch on random data

If you already imported `ncl_diffusion` in this runtime, do **Runtime > Restart session** first.

In [ ]:
# ====== BOOTSTRAP (identical in every notebook) ======
# Mounts Drive, links the project folder, installs anything missing, puts the package on sys.path.
import os, sys, importlib.util, importlib.metadata, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules

PROJECT = "/content/drive/MyDrive/ncl_diffusion" if IN_COLAB else os.path.abspath("ct_diffusion_project")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(PROJECT, exist_ok=True)
LINK = "/content/ncl_diffusion_project" if IN_COLAB else PROJECT   # short path used everywhere below
if LINK != PROJECT and not os.path.exists(LINK):
    os.symlink(PROJECT, LINK)

# pip name -> import name.
# NOTE: no `monai-generative` any more. MONAI >= 1.4 ships the diffusion code in core.
_pip = {"einops": "einops", "pydicom": "pydicom", "kaggle": "kaggle", "scikit-learn": "sklearn", "tqdm": "tqdm",
        "opencv-python-headless": "cv2", "pandas": "pandas", "matplotlib": "matplotlib",
        "torchmetrics": "torchmetrics", "torch-fidelity": "torch_fidelity"}
_missing = [p for p, m in _pip.items() if importlib.util.find_spec(m) is None]
try:
    from packaging.version import Version
    _monai_ok = Version(importlib.metadata.version("monai")) >= Version("1.4")
except Exception:          # not installed, or version string we can't parse
    _monai_ok = False
if not _monai_ok:
    _missing.append("monai>=1.4")
if _missing:
    print("installing:", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
if LINK not in sys.path:
    sys.path.insert(0, LINK)
print("project:", PROJECT)

In [ ]:
# ====== DIRECTORIES (fixed) ======
import shutil
for d in ["ncl_diffusion", "ncl_diffusion/data", "ncl_diffusion/models", "ncl_diffusion/train"]:
    os.makedirs(f"{LINK}/{d}", exist_ok=True)

# remove the stray folder the old typo created (only if it holds no files)
stray = Path(LINK) / "ncl_diffusion,ncl_diffusion"
if stray.exists() and not any(p.is_file() for p in stray.rglob("*")):
    shutil.rmtree(stray); print("removed stray folder", stray)
print("Directories OK.")

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/compat.py
"""Single import point for the MONAI diffusion components.
MONAI >= 1.4 has them in core. Falls back to the old `generative` package if an older MONAI is installed."""
try:
    from monai.networks.nets import DiffusionModelUNet
    from monai.networks.schedulers import DDPMScheduler, DDIMScheduler
    SOURCE = "monai-core"
except ImportError:  # pragma: no cover
    from generative.networks.nets import DiffusionModelUNet
    from generative.networks.schedulers import DDPMScheduler, DDIMScheduler
    SOURCE = "monai-generative"

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/model_imports.py
"""Kept for backwards compatibility: everything now lives in compat.py."""
from .compat import DiffusionModelUNet, DDPMScheduler, DDIMScheduler, SOURCE  # noqa: F401

In [ ]:
# ====== PATCH config.py: add a GanCfg section (skipped if already patched) ======
GAN_CFG_SRC = """@dataclass
class GanCfg:
    # --- architecture ---
    z_dim: int = 128
    cond_dim: int = 128               # size of the label embedding fed to G (conditional norm layers)
    g_ch: int = 64                    # base width of G (x8 at the 4x4 level)
    d_ch: int = 64
    # --- optimisation (TTUR: D learns faster than G) ---
    lr_g: float = 1e-4
    lr_d: float = 4e-4
    beta1: float = 0.0
    beta2: float = 0.99
    d_steps: int = 1                  # D updates per G update
    ema_decay: float = 0.999          # EMA of G weights (used for sampling)
    augment: list = field(default_factory=lambda: ["translation", "cutout"])  # DiffAugment policy, geometry only
    # --- schedule ---
    run_name: str = "cgan_base"
    epochs: int = 100
    batch_size: int = 32
    ckpt_every: int = 5
    sample_every: int = 5
    n_preview: int = 24
"""

p = Path(LINK) / "ncl_diffusion" / "config.py"
assert p.exists(), "config.py not found. Run Project_setup.ipynb first."
s = p.read_text()
if "class GanCfg" in s:
    print("config.py already patched")
else:
    a_sections = '_SECTIONS = ("paths", "data", "diffusion", "train", "sample", "clf")'
    a_field = "    clf: ClfCfg = field(default_factory=ClfCfg)"
    assert a_sections in s and a_field in s, "config.py differs from what I expected. Add the GanCfg section by hand."
    s = s.replace(a_sections, GAN_CFG_SRC + "\n\n" + a_sections.replace('"clf")', '"clf", "gan")'))
    s = s.replace(a_field, a_field + "\n    gan: GanCfg = field(default_factory=GanCfg)")
    p.write_text(s)
    print("config.py patched")

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/models/gan.py
"""Label-conditional GAN baseline for the diffusion-vs-GAN comparison.

ResNet generator with class-conditional GroupNorm, projection discriminator (handles multi-hot labels),
hinge loss, spectral norm on D, DiffAugment. GroupNorm instead of BatchNorm keeps train/eval behaviour identical,
so the EMA generator needs no running-stat fix.

GANSystem mirrors DiffusionSystem (.sample(), .in_channels, .img_size) so synth.generate_synthetic, the
downstream experiments and the metrics work on both generators unchanged."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm as SN


# ----------------------------------------------------------------------------- generator
class CondGroupNorm(nn.Module):
    """GroupNorm (no affine) followed by a per-channel scale/shift predicted from the label embedding."""
    def __init__(self, ch, cond_dim, groups=32):
        super().__init__()
        self.norm = nn.GroupNorm(min(groups, ch), ch, affine=False)
        self.gamma, self.beta = nn.Linear(cond_dim, ch), nn.Linear(cond_dim, ch)
        nn.init.zeros_(self.gamma.weight); nn.init.ones_(self.gamma.bias)    # start as scale 1, shift 0
        nn.init.zeros_(self.beta.weight); nn.init.zeros_(self.beta.bias)

    def forward(self, x, c):
        return self.norm(x) * self.gamma(c)[:, :, None, None] + self.beta(c)[:, :, None, None]


class GBlock(nn.Module):
    """Pre-activation residual block that doubles the resolution."""
    def __init__(self, cin, cout, cond_dim):
        super().__init__()
        self.n1, self.n2 = CondGroupNorm(cin, cond_dim), CondGroupNorm(cout, cond_dim)
        self.c1, self.c2 = nn.Conv2d(cin, cout, 3, padding=1), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1)

    def forward(self, x, c):
        h = F.interpolate(F.relu(self.n1(x, c)), scale_factor=2, mode="nearest")
        h = self.c2(F.relu(self.n2(self.c1(h), c)))
        return h + self.skip(F.interpolate(x, scale_factor=2, mode="nearest"))


class Generator(nn.Module):
    def __init__(self, n_labels, out_channels=1, img_size=128, z_dim=128, ch=64, cond_dim=128):
        super().__init__()
        n_up = int(math.log2(img_size // 4))
        assert 4 * 2 ** n_up == img_size, "img_size must be 4 * 2^k (64, 128, 256, ...)"
        w = [ch * min(8, 2 ** max(n_up - 1 - i, 0)) for i in range(n_up + 1)]   # 128px, ch=64: 512,512,256,128,64,64
        self.z_dim, self.w0 = z_dim, w[0]
        self.embed = nn.Sequential(nn.Linear(n_labels, cond_dim), nn.SiLU())     # multi-hot labels -> condition vector
        self.fc = nn.Linear(z_dim + cond_dim, w[0] * 16)                         # -> 4x4 feature map
        self.blocks = nn.ModuleList([GBlock(w[i], w[i + 1], cond_dim) for i in range(n_up)])
        self.out = nn.Sequential(nn.GroupNorm(min(32, w[-1]), w[-1]), nn.ReLU(),
                                 nn.Conv2d(w[-1], out_channels, 3, padding=1), nn.Tanh())

    def forward(self, z, y):
        c = self.embed(y)
        h = self.fc(torch.cat([z, c], 1)).view(-1, self.w0, 4, 4)
        for b in self.blocks:
            h = b(h, c)
        return self.out(h)


# ----------------------------------------------------------------------------- discriminator
class DBlock(nn.Module):
    def __init__(self, cin, cout, down=True, first=False):
        super().__init__()
        self.c1, self.c2 = SN(nn.Conv2d(cin, cout, 3, padding=1)), SN(nn.Conv2d(cout, cout, 3, padding=1))
        self.skip = SN(nn.Conv2d(cin, cout, 1))
        self.down, self.first = down, first

    def forward(self, x):
        h = x if self.first else F.leaky_relu(x, 0.2)
        h = self.c2(F.leaky_relu(self.c1(h), 0.2))
        s = self.skip(x)
        if self.down:
            h, s = F.avg_pool2d(h, 2), F.avg_pool2d(s, 2)
        return h + s


class Discriminator(nn.Module):
    """Projection discriminator: score = linear(features) + <embed(labels), features>."""
    def __init__(self, n_labels, in_channels=1, img_size=128, ch=64):
        super().__init__()
        n_down = int(math.log2(img_size // 4))
        w = [ch * min(8, 2 ** i) for i in range(n_down + 1)]                     # 128px, ch=64: 64,128,256,512,512,512
        blocks = [DBlock(in_channels, w[0], first=True)]                         # img_size -> img_size/2
        blocks += [DBlock(w[i], w[i + 1]) for i in range(n_down - 1)]            # ... -> 4x4
        blocks += [DBlock(w[n_down - 1], w[n_down - 1], down=False)]             # refine at 4x4
        self.blocks = nn.Sequential(*blocks)
        feat = w[n_down - 1]
        self.fc = SN(nn.Linear(feat, 1))
        self.proj = SN(nn.Linear(n_labels, feat, bias=False))

    def forward(self, x, y):
        h = F.leaky_relu(self.blocks(x), 0.2).sum((2, 3))
        return self.fc(h).squeeze(1) + (self.proj(y) * h).sum(1)


# ----------------------------------------------------------------------------- DiffAugment
def diff_augment(x, policy=("translation", "cutout")):
    """Differentiable augmentation applied to BOTH real and fake images before D (Zhao et al. 2020); it stops D
    from memorising a small training set. Geometry only (no brightness/contrast) because intensities carry
    meaning in CT. Empty regions are filled with background (-1 = black)."""
    B, _, H, W = x.shape
    if "translation" in policy:
        shift = (torch.rand(B, 2, device=x.device) * 2 - 1) * 0.25               # +-0.25 in [-1,1] coords = +-12.5% of size
        theta = torch.zeros(B, 2, 3, device=x.device, dtype=x.dtype)
        theta[:, 0, 0] = 1
        theta[:, 1, 1] = 1
        theta[:, :, 2] = shift.to(x.dtype)
        grid = F.affine_grid(theta, list(x.shape), align_corners=False)
        x = F.grid_sample(x + 1, grid, padding_mode="zeros", align_corners=False) - 1
    if "cutout" in policy:
        cy = torch.randint(0, H, (B,), device=x.device)
        cx = torch.randint(0, W, (B,), device=x.device)
        ys = torch.arange(H, device=x.device)[None, :, None]
        xs = torch.arange(W, device=x.device)[None, None, :]
        hole = ((ys - cy[:, None, None]).abs() < H // 4) & ((xs - cx[:, None, None]).abs() < W // 4)   # ~half-size square
        x = (x + 1) * (~hole)[:, None].to(x.dtype) - 1
    return x


# ----------------------------------------------------------------------------- system
class GANSystem(nn.Module):
    def __init__(self, cfg, in_channels, n_labels):
        super().__init__()
        g = cfg.gan
        self.in_channels, self.img_size, self.z_dim = in_channels, cfg.data.img_size, g.z_dim
        self.G = Generator(n_labels, in_channels, self.img_size, g.z_dim, g.g_ch, g.cond_dim)
        self.D = Discriminator(n_labels, in_channels, self.img_size, g.d_ch)

    @property
    def device(self):
        return next(self.parameters()).device

    @torch.no_grad()
    def sample(self, labels=None, n=None, generator=None, **_):
        """Same call as DiffusionSystem.sample; extra kwargs (steps, cfg_scale, sampler) are ignored. Returns [-1,1]."""
        dev = self.device
        n = labels.shape[0] if labels is not None else n
        z = torch.randn(n, self.z_dim, device=dev, generator=generator)
        return self.G(z, labels.to(dev).float()).clamp(-1, 1)

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/train/gan.py
"""GAN training loop: hinge loss, TTUR, EMA generator, DiffAugment.
Checkpoint / resume / preview conventions match train/diffusion.py."""
from pathlib import Path
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from ..models.gan import diff_augment, GANSystem
from ..utils import to_model_range, to_unit_range, show_grid
from .ema import EMA


def train_gan(system, train_loader, cfg, run_dir, device="cuda", preview_labels=None, eval_fn=None, eval_every=0):
    """train_loader should use drop_last=True. There is no validation loss for a GAN, so checkpoints are chosen
    with eval_fn(system) -> dict of metrics (called with EMA weights loaded; e.g. {"fid": ...}).
    If it returns a 'fid' key the lowest-FID model is also saved as ckpt_best.pt."""
    gc, run_dir = cfg.gan, Path(run_dir)
    (run_dir / "samples").mkdir(parents=True, exist_ok=True)
    system.to(device)
    G, D, policy = system.G, system.D, tuple(gc.augment)
    optG = torch.optim.Adam(G.parameters(), lr=gc.lr_g, betas=(gc.beta1, gc.beta2))
    optD = torch.optim.Adam(D.parameters(), lr=gc.lr_d, betas=(gc.beta1, gc.beta2))
    ema = EMA(G, gc.ema_decay)
    start, history, best = 0, [], float("inf")

    def pack(epoch, with_opt):
        ck = {"G": G.state_dict(), "D": D.state_dict(), "ema": ema.state_dict(), "epoch": epoch,
              "history": history, "best": best}
        if with_opt:
            ck.update(optG=optG.state_dict(), optD=optD.state_dict())
        return ck

    last = run_dir / "ckpt_last.pt"
    if last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"]); ema.load_state_dict(ck["ema"])
        start, history, best = ck["epoch"] + 1, ck["history"], ck["best"]
        print(f"resumed from epoch {start}")

    for ep in range(start, gc.epochs):
        G.train(); D.train(); run = {"lossD": [], "lossG": [], "d_real": [], "d_fake": []}
        for batch in tqdm(train_loader, desc=f"epoch {ep+1}/{gc.epochs}", leave=False):
            x = to_model_range(batch["image"].to(device, non_blocking=True))
            y = batch["label"].to(device, non_blocking=True).float()
            B = x.shape[0]
            # ---- discriminator: hinge loss on augmented real / fake
            for _ in range(gc.d_steps):
                with torch.no_grad():
                    fake = G(torch.randn(B, gc.z_dim, device=device), y)
                d_real = D(diff_augment(x, policy), y)
                d_fake = D(diff_augment(fake, policy), y)
                lossD = F.relu(1 - d_real).mean() + F.relu(1 + d_fake).mean()
                optD.zero_grad(set_to_none=True); lossD.backward(); optD.step()
            # ---- generator
            D.requires_grad_(False)
            lossG = -D(diff_augment(G(torch.randn(B, gc.z_dim, device=device), y), policy), y).mean()
            optG.zero_grad(set_to_none=True); lossG.backward(); optG.step()
            D.requires_grad_(True)
            ema.update(G)
            for k, v in zip(run, (lossD, lossG, d_real.mean(), d_fake.mean())):
                run[k].append(v.item())
        row = {"epoch": ep + 1, **{k: sum(v) / len(v) for k, v in run.items()}}

        if eval_fn is not None and eval_every and ((ep + 1) % eval_every == 0 or ep + 1 == gc.epochs):
            with ema.average_parameters(G):
                G.eval()
                row.update(eval_fn(system))
            if "fid" in row and row["fid"] < best:
                best = row["fid"]
                torch.save(pack(ep, False), run_dir / "ckpt_best.pt")
        history.append(row); print({k: round(v, 4) for k, v in row.items()})

        if preview_labels is not None and ((ep + 1) % gc.sample_every == 0 or ep + 1 == gc.epochs):
            with ema.average_parameters(G):
                x = system.sample(torch.as_tensor(preview_labels[:gc.n_preview]).float().to(device),
                                  generator=torch.Generator(device).manual_seed(0))
            show_grid(to_unit_range(x).cpu(), nrow=8, title=f"EMA samples, epoch {ep+1}",
                      save=run_dir / "samples" / f"ep{ep+1:03d}.png")
        if (ep + 1) % gc.ckpt_every == 0 or ep + 1 == gc.epochs:
            torch.save(pack(ep, True), last)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    return pd.DataFrame(history)


def load_gan(cfg, run_dir, in_channels, n_labels, device="cuda", which="last", use_ema=True):
    """Rebuild the GAN from cfg and load EMA (or raw) generator weights from ckpt_{which}.pt."""
    system = GANSystem(cfg, in_channels, n_labels)
    ck = torch.load(Path(run_dir) / f"ckpt_{which}.pt", map_location="cpu", weights_only=False)
    system.G.load_state_dict(ck["ema"]["shadow"] if use_ema else ck["G"])
    system.D.load_state_dict(ck["D"])
    return system.to(device).eval()

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/models/__init__.py
from . import conditioners, backbones, heads, classifiers, gan  # noqa: F401  (register components)
from .system import DiffusionSystem, FeatureRecorder, build_scheduler
from .gan import GANSystem

In [ ]:
%%writefile /content/ncl_diffusion_project/ncl_diffusion/train/__init__.py
from .ema import EMA
from .diffusion import train_diffusion, load_system
from .classifier import train_classifier, evaluate, predict
from .gan import train_gan, load_gan

## Smoke test
Builds tiny versions of both models, trains 1 epoch each on random images, then reloads the checkpoints and
samples. This checks the package imports, the MONAI API calls, the training loops and checkpoint loading.
If anything fails here, paste the traceback back to me. It takes about a minute on a GPU.

In [ ]:
import importlib, tempfile, shutil
import numpy as np, pandas as pd, torch
importlib.invalidate_caches()
from ncl_diffusion.config import Config
from ncl_diffusion import compat, data, models, train          # importing these registers every component
from ncl_diffusion.registry import DATASETS, BACKBONES, CONDITIONERS, CLASSIFIERS
from ncl_diffusion.data.datasets import CachedArrayDataset, make_loader
from ncl_diffusion.models import DiffusionSystem, GANSystem
from ncl_diffusion.train import train_diffusion, train_gan, load_system, load_gan

print("MONAI diffusion components from:", compat.SOURCE)
print(DATASETS, BACKBONES, CONDITIONERS, CLASSIFIERS, sep="\n")

dev = "cuda" if torch.cuda.is_available() else "cpu"
tmp = tempfile.mkdtemp()
cfg = Config().override({
    "paths.project": tmp, "data.img_size": 64, "data.num_workers": 0,
    "diffusion.num_channels": [32, 64, 64],
    "train.epochs": 1, "train.ckpt_every": 1, "train.sample_every": 1, "train.n_preview": 8,
    "gan.epochs": 1, "gan.ckpt_every": 1, "gan.sample_every": 1, "gan.g_ch": 16, "gan.d_ch": 16, "gan.n_preview": 8,
})
lc = cfg.data.label_cols

# random "cache": 64 uint8 images + random multi-hot labels
imgs = np.random.randint(0, 256, (64, 1, 64, 64), dtype=np.uint8)
man = pd.DataFrame(np.random.randint(0, 2, (64, len(lc))), columns=lc); man["idx"] = np.arange(64)
loader = make_loader(CachedArrayDataset(imgs, man, lc), 16, True, 0, drop_last=True)
labels = man[lc].to_numpy(np.float32)

# --- diffusion
run = cfg.run_dir("smoke_ddpm")
train_diffusion(DiffusionSystem(cfg, 1, len(lc)), loader, loader, cfg, run, dev, preview_labels=labels)
ds = load_system(cfg, run, 1, len(lc), dev)
print("diffusion sample:", tuple(ds.sample(torch.tensor(labels[:4]), steps=3).shape))

# --- GAN
run = cfg.run_dir("smoke_gan")
train_gan(GANSystem(cfg, 1, len(lc)), loader, cfg, run, dev, preview_labels=labels)
gs = load_gan(cfg, run, 1, len(lc), dev)
print("GAN sample:      ", tuple(gs.sample(torch.tensor(labels[:4])).shape))

shutil.rmtree(tmp)
print("\nSMOKE TEST PASSED")

In [ ]:
# ====== re-save config.json so it now contains the `gan` section ======
from ncl_diffusion.config import Config
cfg_path = Path(LINK) / "config.json"
cfg = Config.load(cfg_path) if cfg_path.exists() else Config()
cfg.paths.project = LINK
print("saved", cfg.save(), "| gan section:", cfg.gan.run_name)

**Next:** `01_data_rsna.ipynb` (download a subset, build the cache, sanity-check the CT preprocessing).